# Flatand Graph Data Import Pipeline

## Approach Sketch
To be implemented...
- one node per outgoing edge, label with where it comes from
- look at switches: at fixed distance which 2 are closest -> suggest using GeoPandas API for this.
- add actions as edge attributes (LFR disjointly to all outgoing)
- export enhanced osmnx 
- write importer for enhanced osmnx to GraphRailEnv -> describe what we expect as input, which attributes etc.
- timetable for graph env
- unit test with policy reaching target (possibly hard-coded actions)
- snaphost-visualization (preferrably one osmnx export with railway features -> could be a moving house or roundabout if there is no train feature...)


## Documentation
### OSMnx and GeoPandas API documentation
* https://osmnx.readthedocs.io/en/stable/user-reference.html
* https://github.com/gboeing/osmnx-examples
* https://geopandas.org/en/stable/docs/user_guide.html

### OSM Railway documentation
* https://wiki.openstreetmap.org/wiki/Railways
* https://wiki.openstreetmap.org/wiki/Railway_stations
* https://wiki.openstreetmap.org/wiki/Tag:railway=signal
* https://wiki.openstreetmap.org/wiki/Tag:railway=switch
* https://wiki.openstreetmap.org/wiki/Tag:railway=rail
* https://wiki.openstreetmap.org/wiki/Tag:route%3Drailway


### OpenRailwayMap -> inspect Railway features
* https://openrailwaymap.app/#view=18.86/47.1194931/9.3132586&style=signals

### Insepct OSM by Node ID / Way ID
* https://www.openstreetmap.org/node/2048248546/history/4
* https://www.openstreetmap.org/way/194299504

In [ ]:
# !python -m pip install jupyter osmnx matplotlib  "folium>=0.12" matplotlib mapclassify
import osmnx as ox

ox.settings.log_console = True
ox.__version__

In [ ]:
# ox.settings.overpass_url='http://localhost:12347'
# ox.settings.overpass_rate_limit=False

In [ ]:
place = "Walenstadt"
# tiles = "cartodbpositron"
mk = {"radius": 6}
# cols = ["height", "addr:housenumber", "addr:street", "addr:postcode"]

## Download OSMnx graph

In [ ]:
# TODO remove
ox.settings.all_oneway = True
G = ox.graph.graph_from_place(
   place,
   retain_all=False, # If True, return the entire graph even if it is not connected. If False, retain only the largest weakly connected component.
   truncate_by_edge=True, # If True, retain nodes the outside bounding box if at least one of the node’s neighbors lies within the bounding box.
   simplify=True, # If True, simplify graph topology with the simplify_graph function.
   custom_filter='["railway"~"rail"]',
)
fig, ax = ox.plot.plot_graph(G, node_size=0, edge_color="w", edge_linewidth=0.2)

## Switch-switch edges: recover one discarded intermediate node

`simplify_graph` collapses every chain of interstitial (non-intersection) nodes
into a single edge between the two real intersections/dead-ends it connects. If
two switches end up directly adjacent in the simplified graph, any interstitial
nodes that originally separated them (degree-2 vertices just shaping the curve)
were silently discarded - we only kept their coordinates in the edge's `geometry`.
A direct switch-to-switch edge is exactly the case our doubled-graph construction
(see the GGG sections below) can't always represent consistently (the crossover and
triangular-junction cases). If such an edge actually had an interstitial node in the
non-simplified graph, keeping just one of them back turns the direct switch-switch
adjacency into switch-plain-switch, which is always representable.

In [ ]:
# a node's OSM railway=switch tag can lag behind the actual topology (see the
# switch_attr_low_degree check earlier in the pipeline) - a true switch is one
# with in+out degree > 2, so use that instead of the raw tag
switches_in_G = {n for n in G.nodes if G.in_degree(n) + G.out_degree(n) > 2}
switch_switch_edges = [
    (u, v) for u, v in G.edges()
    if u in switches_in_G and v in switches_in_G
]
print(f"found {len(switch_switch_edges)} switch-switch edge(s) in the simplified graph:")
for u, v in switch_switch_edges:
    print(f"  {u} -> {v}")


In [ ]:
# same download parameters as above, but unsimplified, so we can see which
# interstitial nodes simplify_graph removed between any two switches
G_non_simple = ox.graph.graph_from_place(
    place,
    retain_all=False,
    truncate_by_edge=True,
    simplify=False,
    custom_filter='["railway"~"rail"]',
)


In [ ]:
# reuse osmnx's own endpoint/path-building logic (see simplify_graph's source)
# to recover exactly the interstitial path it would collapse between each pair
# of nodes in the unsimplified graph, and check it against our switch-switch edges
from osmnx.simplification import _get_paths_to_simplify

paths_in_non_simple = {
    (path[0], path[-1]): path
    for path in _get_paths_to_simplify(G_non_simple, None, None)
}

for u, v in switch_switch_edges:
    path = paths_in_non_simple.get((u, v)) or paths_in_non_simple.get((v, u))
    if path is None:
        # _get_paths_to_simplify() only yields a path when the edge needs
        # simplifying - if both endpoints are already (true, degree>2)
        # endpoints with nothing interstitial between them, it's a genuinely
        # direct edge in the raw, non-simplified data too
        assert G_non_simple.has_edge(u, v) or G_non_simple.has_edge(v, u), (
            f"edge {u}-{v} not found at all in G_non_simple"
        )
        print(f"{u} -> {v}: 0 intermediate nodes - already directly adjacent in the non-simplified graph too")
        continue
    intermediate = path[1:-1]
    print(f"{u} -> {v}: {len(intermediate)} intermediate node(s) in G_non_simple: {intermediate}")


In [ ]:
# Custom simplification: identical to osmnx.simplification.simplify_graph (see
# its source - _is_endpoint/_get_paths_to_simplify/_build_path determine the
# endpoints and interstitial paths; simplify_graph then aggregates each path's
# edge attributes - summing "length"/"travel_time", keeping a single value
# where all segments agree, else keeping the list of unique values - and
# builds the merged edge's `geometry` from every node's coordinates along the
# path), EXCEPT: whenever a path's two endpoints are both switches and it has
# at least one interstitial node, that node is kept instead of removed, and
# the path is built as two edges (through that node) instead of one - using
# the exact same per-path attribute-aggregation/geometry logic, just applied
# to each half separately.
# NOTE: this does not guarantee an intermediate node survives between every
# pair of switches - if two switches are already directly adjacent in
# G_non_simple (0 interstitial nodes for that path), there is nothing to keep,
# and the switch-switch edge remains direct. Whether that happens is purely a
# property of the OSM data (how finely the way was split), not something this
# function controls.

import itertools

from shapely import LineString, Point


def simplify_graph_keep_switch_intermediate(G, switch_osmids, edge_attr_aggs=None):
    if G.graph.get("simplified"):
        raise ValueError("This graph has already been simplified, cannot simplify it again.")
    if edge_attr_aggs is None:
        edge_attr_aggs = {"length": sum, "travel_time": sum}

    G = G.copy()

    def path_edge_attrs(path):
        """Aggregate a path's edge attributes and build its geometry, exactly
        as simplify_graph does for one interstitial path."""
        path_attributes = {}
        for u, v in itertools.pairwise(path):
            edge_data = next(iter(G.get_edge_data(u, v).values()))
            for attr, value in edge_data.items():
                path_attributes.setdefault(attr, []).append(value)
        for attr_name, attr_values in path_attributes.items():
            if attr_name in edge_attr_aggs:
                path_attributes[attr_name] = edge_attr_aggs[attr_name](attr_values)
            elif len(set(attr_values)) == 1:
                path_attributes[attr_name] = attr_values[0]
            else:
                path_attributes[attr_name] = list(set(attr_values))
        path_attributes["geometry"] = LineString(
            [Point((G.nodes[n]["x"], G.nodes[n]["y"])) for n in path]
        )
        return path_attributes

    all_nodes_to_remove = []
    all_edges_to_add = []

    for path in _get_paths_to_simplify(G, None, None):
        is_switch_switch = path[0] in switch_osmids and path[-1] in switch_osmids
        if is_switch_switch and len(path) > 2:
            # keep exactly one interstitial node - split the path in two there
            mid_idx = len(path) // 2
            mid_node = path[mid_idx]
            for sub_path in (path[: mid_idx + 1], path[mid_idx:]):
                all_edges_to_add.append({
                    "origin": sub_path[0],
                    "destination": sub_path[-1],
                    "attr_dict": path_edge_attrs(sub_path),
                })
            all_nodes_to_remove.extend(n for n in path[1:-1] if n != mid_node)
        else:
            all_edges_to_add.append({
                "origin": path[0],
                "destination": path[-1],
                "attr_dict": path_edge_attrs(path),
            })
            all_nodes_to_remove.extend(path[1:-1])

    for edge in all_edges_to_add:
        G.add_edge(edge["origin"], edge["destination"], **edge["attr_dict"])
    G.remove_nodes_from(set(all_nodes_to_remove))
    G.graph["simplified"] = True
    return G


# same degree-based definition as above (not the raw OSM tag), computed on
# G_non_simple directly - its real branch points still have degree > 2
switch_osmids = {n for n in G_non_simple.nodes if G_non_simple.in_degree(n) + G_non_simple.out_degree(n) > 2}
G_fixed = simplify_graph_keep_switch_intermediate(G_non_simple, switch_osmids)

switches_in_G_fixed = {n for n in G_fixed.nodes if G_fixed.in_degree(n) + G_fixed.out_degree(n) > 2}
fixed_switch_switch_edges = [
    (u, v) for u, v in G_fixed.edges()
    if u in switches_in_G_fixed and v in switches_in_G_fixed
]
print(f"switch-switch edges remaining after the fix: {fixed_switch_switch_edges}")


In [ ]:
G = G_fixed

In [ ]:
gdf = ox.features.features_from_place(place, 
                                      tags={"building": True}
                                     )
m = gdf.explore(
    #tiles=tiles,
    #tooltip=cols
)
nodes,edges = ox.convert.graph_to_gdfs(G)

In [ ]:
edges.explore(m=m, color="purple")

In [ ]:
nodes.explore(marker_kwds={"radius": 8})

In [ ]:
list(G.nodes(data=True))

## Visualize networkx graph

In [ ]:
import networkx as nx
nx.draw(G, labels={n[0]: str(n[0]) for n in list(G.nodes(data=True))})

## Stations

In [ ]:
stations = ox.features.features_from_place(place, tags={"railway": "station"})
stations.explore(m=m, 
                 #tiles=tiles, 
                 color="yellow", tooltip="name", marker_kwds=mk)

## Signals

In [ ]:
signals = ox.features.features_from_place(place, tags={"railway": "signal"})

In [ ]:
m = gdf.explore(
    #tiles=tiles,
    #tooltip=cols
)
nodes,edges = ox.convert.graph_to_gdfs(G)
nodes.explore(m=m, color="purple")

## Export graph to graphml

In [ ]:
# TODO must not be simplified - alternative?
ox.io.save_graphml(G, filepath="walensee.graphml")

## Import graph from graphml

In [ ]:
GG = ox.io.load_graphml(filepath="walensee.graphml")

In [ ]:
# TODO MultiDiGraph - due to non-simplify?
GG = nx.DiGraph(GG)

## Graph Rail Env

In [ ]:
for n, d in GG.nodes(data=True):
    if "railway" not in d or d["railway"] != "switch":
        continue
    print(type(n))
    print(n,d)

In [ ]:
SWITCH=2042222936

In [ ]:
for n in list(GG.successors(SWITCH)):
    print(n, GG.get_edge_data(SWITCH, n, default=None))

In [ ]:
TARGET=2042193124

In [ ]:
for n in list(GG.predecessors(SWITCH)):
    print(n, GG.get_edge_data(n, SWITCH, default=None))

In [ ]:
#!python -m pip install flatland-rl==4.3.0
from flatland.core.env_observation_builder import DummyObservationBuilder
from flatland.envs.graph.rail_graph_transition_map import GraphTransitionMap
from flatland.core.graph.graph_resource_map import GraphResourceMap
from flatland.envs.graph.distance_map import GraphDistanceMap
from flatland.envs.graph_rail_env import GraphRailEnv
from flatland.envs.timetable_utils import Timetable
from flatland.envs import malfunction_generators as mal_gen
from flatland.envs.rewards import DefaultRewards
from flatland.envs.rail_env import RailEnvActions

In [ ]:
# add actions
for v1,v2, edge_data in GG.edges(data=True):
    # TODO actions on outgoing edge data plus v1's prohibited_actions attribute should partition all actions 
    edge_data["actions"] = {RailEnvActions.MOVE_FORWARD}
    # TODO do we still need it?
    edge_data["straight"] = v2
    #GG.edges(v1,v2)["actions"]= {RailEnvActions.MOVE_FORWARD}
    # print(v1,v2,edge_data)

In [ ]:
resource_map = {}
for n1, n2 in GG.edges:
    resource_map[n1] = str(sorted([n1,n2]))
# print(resource_map)

agent_waypoints = {
    0: [[SWITCH], [TARGET]]
}
timetable = Timetable(
    earliest_departures=[[0,None]],
    latest_arrivals=[[None,5]],
    max_episode_steps=99,
)

agent_speeds = {
    0: 1/2
}

graph_env = GraphRailEnv.from_graph(
    g=GG,
    resource_map=resource_map,
    agent_waypoints=agent_waypoints,
    agent_speeds=agent_speeds,
    observation_builder=DummyObservationBuilder(),
    malfunction_generator=mal_gen.NoMalfunctionGen(),
    timetable_generator=lambda *args, **kwargs: timetable,
    seed=42,
    rewards=DefaultRewards(),
)
graph_env.reset()

In [ ]:
GG.get_edge_data(SWITCH,TARGET)

In [ ]:
graph_env.rail.g.get_edge_data(SWITCH,TARGET)

In [ ]:
graph_env.reset()
for _ in range(99):
    print("==============================")
    print(f"elapsed before step {graph_env._elapsed_steps}")
    for agent in graph_env.agents:
        print(f"agent.initial_configuration={agent.initial_configuration}")
        print(f"agent.current_configuration={agent.current_configuration}")
        print(f"agent.targets={agent.targets}")
        print(f"agent.speed_counter.speed={agent.speed_counter.speed}")
        print(f"agent.speed_counter.max_speed={agent.speed_counter.max_speed}")
        print(f"agent.state={agent.state}")
        
    graph_env.step({0: RailEnvActions.MOVE_FORWARD})
    if graph_env.dones["__all__"]:
        print("==============================")
        print("done")
        break

## Switch geometry: which two legs stay closest?

For the switch node `SWITCH` (from the osmnx graph `GG` loaded back from `walensee.graphml`) we walk
`DISTANCE` metres along each of its three incident edges (starting at the switch) and compare the
resulting probe points pairwise: the pair that stays closest together is the one that continues
through the switch in (nearly) the same direction, the remaining leg is the odd one out.

In [ ]:
import itertools

import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString, Point

DISTANCE = 10  # metres along the edge, measured from the switch


def oriented_geometry(graph, u, v, k, start):
    """Geometry of edge (u, v, k) as a LineString that starts at node `start`."""
    geom = graph.edges[u, v, k].get("geometry")
    if geom is None:
        # unsimplified graph: the edge is the straight segment between its two nodes
        geom = LineString([(graph.nodes[u]["x"], graph.nodes[u]["y"]),
                           (graph.nodes[v]["x"], graph.nodes[v]["y"])])
    start_point = Point(graph.nodes[start]["x"], graph.nodes[start]["y"])
    if start_point.distance(Point(geom.coords[-1])) < start_point.distance(Point(geom.coords[0])):
        geom = geom.reverse()
    return geom


# GG was downcast to a plain DiGraph above (for the flatland env), but osmnx's
# projection/plotting helpers need a MultiDiGraph - rebuild one just for this section
GG_multi = nx.MultiDiGraph(GG)

# project to UTM so that lengths and interpolation are in metres
G_proj = ox.projection.project_graph(GG_multi)

# all edges incident to the switch, no matter their direction in the digraph
incident = ([(SWITCH, v, k, v) for _, v, k in G_proj.out_edges(SWITCH, keys=True)]
            + [(u, SWITCH, k, u) for u, _, k in G_proj.in_edges(SWITCH, keys=True)])

probes = {}
for u, v, k, neighbour in incident:
    geom = oriented_geometry(G_proj, u, v, k, SWITCH)
    if geom.length < DISTANCE:
        print(f"edge {SWITCH} -> {neighbour} is only {geom.length:.1f} m long, "
              f"probe clipped to its far end")
    probes[neighbour] = geom.interpolate(DISTANCE)

probe_points = gpd.GeoDataFrame({"neighbour": list(probes)},
                                geometry=list(probes.values()),
                                crs=G_proj.graph["crs"])
probe_points

In [ ]:
pairs = pd.DataFrame(
    [(a, b, pa.distance(pb)) for (a, pa), (b, pb) in itertools.combinations(probes.items(), 2)],
    columns=["neighbour_1", "neighbour_2", "distance"],
).sort_values("distance", ignore_index=True)
display(pairs)

closest = pairs.iloc[0]
print(f"at {DISTANCE} m from {SWITCH} the closest pair is "
      f"{closest.neighbour_1} / {closest.neighbour_2} ({closest.distance:.3f} m apart); "
      f"odd one out: {set(probes) - {closest.neighbour_1, closest.neighbour_2}}")

In [ ]:
# visual check: the switch (blue), its three incident edges (green), the three original
# neighbours (orange) and the three probe points at DISTANCE along each edge (red)
ego = nx.ego_graph(G_proj, SWITCH, radius=1, undirected=True)
ego_nodes, ego_edges = ox.convert.graph_to_gdfs(ego)
m_switch = ego_edges.to_crs(epsg=4326).explore(color="green")
# ego_nodes is indexed by osmid (not a column); explore() resets the index itself
# and surfaces it as an "osmid" tooltip column, so don't add that column again here
ego_nodes.loc[[SWITCH]].to_crs(epsg=4326).explore(m=m_switch, color="blue", tooltip="osmid", marker_kwds=mk)
ego_nodes.loc[list(probes)].to_crs(epsg=4326).explore(m=m_switch, color="orange", tooltip="osmid", marker_kwds=mk)
probe_points.to_crs(epsg=4326).explore(m=m_switch, color="red", tooltip="neighbour", marker_kwds=mk)

## Switch geometry as a reusable function

In [ ]:
def closest_neighbors(graph, switch, distance=DISTANCE):
    """Return the two neighbours of `switch` whose edges stay closest together.

    `graph` must be a (multi-)digraph projected to a metric CRS (see
    `ox.projection.project_graph`). We walk `distance` metres along each edge
    incident to `switch` and return the pair of neighbours whose probe points
    end up closest to each other - i.e. the leg pair that continues through
    the switch in (nearly) the same direction. If any incident edge is
    shorter than `distance`, the probing distance is truncated to the
    shortest incident edge's length for ALL of the switch's edges - clipping
    each edge independently would put a short edge's probe much closer to the
    switch than the others, which could make it look artificially "closest"
    to every other neighbour.
    """
    multi_graph = graph if graph.is_multigraph() else nx.MultiDiGraph(graph)

    incident = ([(switch, v, k, v) for _, v, k in multi_graph.out_edges(switch, keys=True)]
                + [(u, switch, k, u) for u, _, k in multi_graph.in_edges(switch, keys=True)])

    geoms = {neighbour: oriented_geometry(multi_graph, u, v, k, switch) for u, v, k, neighbour in incident}
    probe_distance = min([distance] + [geom.length for geom in geoms.values()])
    probes = {neighbour: geom.interpolate(probe_distance) for neighbour, geom in geoms.items()}

    pairs = pd.DataFrame(
        [(a, b, pa.distance(pb)) for (a, pa), (b, pb) in itertools.combinations(probes.items(), 2)],
        columns=["neighbour_1", "neighbour_2", "distance"],
    ).sort_values("distance", ignore_index=True)

    closest = pairs.iloc[0]
    return closest.neighbour_1, closest.neighbour_2


neighbor_1, neighbor_2 = closest_neighbors(G_proj, SWITCH)
print(f"closest neighbours of switch {SWITCH}: {neighbor_1} and {neighbor_2}")

## Sanity check: run `closest_neighbors` on every switch

In [ ]:
switches = [n for n in GG.nodes if GG.in_degree(n) + GG.out_degree(n) > 2]
print(f"found {len(switches)} node(s) with in+out degree > 2: {switches}")

missing_switch_attr = [n for n in switches if GG.nodes[n].get("railway") != "switch"]
assert not missing_switch_attr, (
    f"node(s) with in+out degree > 2 but no switch attribute: {missing_switch_attr}"
)
print("OK: all nodes with in+out degree > 2 have the switch attribute")

switch_attr_low_degree = [
    n for n, d in GG.nodes(data=True)
    if d.get("railway") == "switch" and GG.in_degree(n) + GG.out_degree(n) <= 2
]
print(f"node(s) with switch attribute but in+out degree <= 2: {switch_attr_low_degree}")

for switch in switches:
    try:
        n1, n2 = closest_neighbors(G_proj, switch)
        print(f"switch {switch}: closest neighbours are {n1} and {n2}")
    except Exception as e:
        print(f"switch {switch}: closest_neighbors FAILED with {type(e).__name__}: {e}")

## Visual inspection: switches vs. low-degree switch-tagged nodes

Plot the whole graph, highlighting nodes with in+out degree > 2 (the switches used above) in one
color and nodes that carry the `switch` attribute but have degree <= 2 (flagged by the sanity
check above) in another.

In [ ]:
from matplotlib.lines import Line2D

switches_set = set(switches)
switch_attr_low_degree_set = set(switch_attr_low_degree)

default_color = "#999999"
switch_color = "#1f77b4"
low_degree_switch_color = "#d62728"

node_color = [
    low_degree_switch_color if n in switch_attr_low_degree_set
    else switch_color if n in switches_set
    else default_color
    for n in GG_multi.nodes
]

fig, ax = ox.plot.plot_graph(
    GG_multi,
    node_color=node_color,
    node_size=15,
    edge_color="w",
    edge_linewidth=0.5,
    show=False,
    close=False,
)
ax.legend(
    handles=[
        Line2D([], [], marker="o", linestyle="", color=switch_color, label="switch (in+out degree > 2)"),
        Line2D([], [], marker="o", linestyle="", color=low_degree_switch_color, label="switch attribute, in+out degree <= 2"),
        Line2D([], [], marker="o", linestyle="", color=default_color, label="other node"),
    ],
    loc="lower left",
)

In [ ]:
def explore_categorized(graph):
    """Display `graph` as an interactive map (edges green), coloring nodes by
    switch category: switch (in+out degree > 2), switch attribute but
    in+out degree <= 2, or other."""
    # ox.convert.graph_to_gdfs() needs (u, v, key) edges, i.e. a MultiDiGraph
    multi_graph = graph if graph.is_multigraph() else nx.MultiDiGraph(graph)
    nodes, edges = ox.convert.graph_to_gdfs(multi_graph)

    m = edges.explore(color="green")

    is_switch = nodes.index.isin(switches_set)
    is_low_degree_switch = nodes.index.isin(switch_attr_low_degree_set)
    is_other = ~(is_switch | is_low_degree_switch)

    if is_other.any():
        nodes[is_other].explore(m=m, color=default_color, tooltip="osmid", marker_kwds=mk)
    if is_switch.any():
        nodes[is_switch].explore(m=m, color=switch_color, tooltip="osmid", marker_kwds=mk)
    if is_low_degree_switch.any():
        nodes[is_low_degree_switch].explore(m=m, color=low_degree_switch_color, tooltip="osmid", marker_kwds=mk)

    return m


# same categories as above, but as an interactive map (see "visual check" above)
explore_categorized(GG_multi)

## Sanity check: max degree in the graph

In [ ]:
over_degree = {n: GG.in_degree(n) + GG.out_degree(n) for n in GG.nodes if GG.in_degree(n) + GG.out_degree(n) > 3}
assert not over_degree, f"node(s) with in+out degree > 3: {over_degree}"
print("OK: all nodes in GG have in+out degree <= 3")

## Build GGG: breadth-first traversal, all paths starting at each node

In [ ]:
from collections import deque


# boundary nodes: only incoming or only outgoing edges in GG (not both)
boundary_nodes = {n for n in GG.nodes if (GG.in_degree(n) == 0) != (GG.out_degree(n) == 0)}
print("boundary nodes:", boundary_nodes)


undirected = GG.to_undirected()

# for each switch, the pair of neighbours whose legs continue straight through it
# (see closest_neighbors()) - going directly from one to the other isn't a real
# routing decision, so the BFS below backtracks instead of walking straight through
close_neighbor_pairs = {s: set(closest_neighbors(G_proj, s)) for s in switches_set}

# non-loopy (simple) paths in the undirected version of GG, starting at each boundary node:
# keep extending the path as long as an unvisited neighbor of the current node exists;
# once no such neighbor can be found, the path is done and gets recorded
boundary_paths = []
for b in boundary_nodes:
    paths = []
    queue = deque([[b]])
    while queue:
        path = queue.popleft()
        last = path[-1]
        predecessor = path[-2] if len(path) > 1 else None
        close_pair = close_neighbor_pairs.get(last)
        extended = False
        for neighbor in undirected.neighbors(last):
            if neighbor in path:
                continue
            if close_pair is not None and predecessor in close_pair and neighbor in close_pair:
                # predecessor -> last -> neighbor would pass straight between last's
                # close neighbors - exclude it, backtrack instead
                continue
            queue.append(path + [neighbor])
            extended = True
        if not extended:
            paths.append(path)

    boundary_paths.extend(paths)
    print(b, paths)

# verify: paths are prefix-free
not_prefix_free = [
    p for p in boundary_paths
    if any(p is not q and len(p) <= len(q) and q[:len(p)] == p for q in boundary_paths)
]
assert not not_prefix_free, f"found non-prefix-free path(s) {not_prefix_free}"
print("OK: all boundary-node paths are prefix-free")

# verify: each path ends at another boundary node
bad_paths = [pp for pp in boundary_paths if pp[-1] not in boundary_nodes]
assert not bad_paths, f"path(s) not ending in a boundary node: {bad_paths}"
print("OK: all boundary-node paths end at another boundary node")

In [ ]:
def prime(n: str):
    if n.endswith("_prime"):
        return n.rstrip("_prime")
    else:
        return f"{n}_prime"

In [ ]:
def find_other_neighbor(graph, n, nn):
    """Return all neighbors of `n` in `graph` (an undirected view of GG) other
    than `nn`.

    If `n` is a switch (has 3 undirected neighbors), the result additionally
    excludes any neighbor that forms a close, near-collinear "straight-through"
    pair with `nn` (see closest_neighbors()) - i.e. passing from `nn` straight
    through the switch is excluded.
    """
    neighbors = list(graph.neighbors(n))
    candidates = [x for x in neighbors if x != nn]
    if len(neighbors) == 3:
        close_pair = set(closest_neighbors(G_proj, n))
        if nn in close_pair:
            candidates = [x for x in candidates if x not in close_pair]
    if not candidates:
        raise ValueError(f"no neighbor of {n} other than {nn} satisfies the constraints")
    return candidates

In [ ]:
def verify_ggg_paths(ggg):
    """Verify that for every boundary_paths path x1,x2,x3,... in GG, there is
    a directed x1_,x2_,x3_,... path in `ggg` such that x1_ is one of str(x1)
    or prime(x1), etc."""

    def find_ggg_edge(x1, x2):
        """Return the (id1, id2) GGG identities - each one of str(x)/prime(str(x))
        - such that (id1, id2) is an edge in `ggg` corresponding to the physical
        GG pair (x1, x2); or None if no such edge exists."""
        x1_str, x2_str = str(x1), str(x2)
        for a in (x1_str, prime(x1_str)):
            for b in (x2_str, prime(x2_str)):
                if (a, b) in ggg.edges:
                    return (a, b)
        return None

    def first_missing_edge(path):
        """First sweep over `path`: find the GGG edge corresponding to each
        consecutive pair of physical (int) GG nodes. Second pass: verify the
        resulting edge list actually represents a directed path in `ggg`, i.e.
        for consecutive edges (n1, n2), (n3, n4) in the list, n2 == n3.
        Returns ("no mapping", (x_i, x_{i+1})) for the first pair with no GGG
        edge found (loop 1), or ("inconsistent path", (x_i, x_{i+1})) for the
        first pair whose GGG edge doesn't chain with the previous one (loop 2),
        or None if the whole path checks out."""
        ggg_edges = []
        for x1, x2 in zip(path, path[1:]):
            edge = find_ggg_edge(x1, x2)
            if edge is None:
                return ("no mapping", (x1, x2))
            ggg_edges.append(edge)

        for i, ((n1, n2), (n3, n4)) in enumerate(zip(ggg_edges, ggg_edges[1:])):
            if n2 != n3:
                return ("inconsistent path", (path[i + 1], path[i + 2]))

        return None

    missing = {}
    for p in boundary_paths:
        failure = first_missing_edge(p)
        if failure is not None:
            missing[tuple(p)] = failure
    assert not missing, f"first missing edge per boundary path (reason, edge): {missing}"
    print("OK: every boundary path in GG has a corresponding directed path in GGG")


In [ ]:
def verify_edge_duality(ggg):
    """Verify that (n1, n2) in edges <=> (prime(n2), prime(n1)) in edges, for
    all n1, n2 in `ggg`.nodes (ggg.nodes already contains every physical node
    and its "_prime" counterpart, so this also covers prime(n1), prime(n2)
    pairs without a separate pass)."""
    edges = set(ggg.edges)
    mismatches = [
        (n1, n2)
        for n1 in ggg.nodes
        for n2 in ggg.nodes
        if ((n1, n2) in edges) != ((prime(n2), prime(n1)) in edges)
    ]
    assert not mismatches, f"edge-reversal duality violated for: {mismatches}"
    print("OK: (n1, n2) in edges <=> (prime(n2), prime(n1)) in edges for all n1, n2 in ggg.nodes")


In [ ]:
GGG = nx.DiGraph()
GGG.graph["crs"] = GG.graph.get("crs")


undirected = GG.to_undirected()


def add_oriented_edge(u, v):
    """Add edge u -> v to GGG, plus its reverse-direction mirror prime(v) -> prime(u).
    `u`, `v` may be plain GG node ids or already "_prime"-suffixed identities."""
    u, v = str(u), str(v)
    u_id = int(u[:-len("_prime")]) if u.endswith("_prime") else int(u)
    v_id = int(v[:-len("_prime")]) if v.endswith("_prime") else int(v)
    edge_data = GG.get_edge_data(u_id, v_id) or GG.get_edge_data(v_id, u_id) or {}

    GGG.add_node(u, **GG.nodes[u_id])
    GGG.add_node(v, **GG.nodes[v_id])
    assert u != v, f"self-loop edge {u} -> {v}"
    GGG.add_edge(u, v, **edge_data)

    pu, pv = prime(u), prime(v)
    GGG.add_node(pv, **GG.nodes[v_id])
    GGG.add_node(pu, **GG.nodes[u_id])
    assert pv != pu, f"self-loop edge {pv} -> {pu}"
    GGG.add_edge(pv, pu, **edge_data)


def find_me(en: str, n_str: str):
    """Find which of n_str/prime(n_str) is the identity of the current node
    already connected to `en` (or prime(en)) in GGG - regardless of edge
    direction.

    A direct check for "en feeds into n_str" (en/prime(en) as source,
    n_str/prime(n_str) as target) isn't enough: if instead n_str's identity
    is the one feeding INTO en, that relationship still shows up in GGG, just
    as the automatic mirror edge prime(en) -> prime(n_str) (added for free by
    add_oriented_edge) - so a direction-sensitive check misreads that mirror
    as "en feeds into prime(n_str)" and returns the wrong identity.

    What actually distinguishes the two identities is direction-agnostic:
    - en/prime(en) <-> n_str/prime(n_str) (same variant, either direction) and
      prime(en)/en <-> prime(n_str)/n_str (its mirror) are the SAME fact -
      they always co-occur - and mean n_str (plain) is the one in use.
    - en/prime(en) <-> prime(n_str)/n_str ("crossing" the variants) and its
      own mirror are likewise the same fact, and mean prime(n_str) is in use.
    """
    en, n_str = str(en), str(n_str)
    same_variant = ((en, n_str) in GGG.edges or (n_str, en) in GGG.edges
                     or (prime(en), prime(n_str)) in GGG.edges or (prime(n_str), prime(en)) in GGG.edges)
    crossing = ((en, prime(n_str)) in GGG.edges or (prime(n_str), en) in GGG.edges
                or (prime(en), n_str) in GGG.edges or (n_str, prime(en)) in GGG.edges)
    if same_variant and not crossing:
        return n_str
    elif crossing and not same_variant:
        return prime(n_str)
    elif same_variant and crossing:
        raise ValueError(f"both identities of {n_str} already connect to {en}/{prime(en)} - genuine conflict")
    else:
        raise ValueError(f"no edge found connecting {en}/{prime(en)} to {n_str}/{prime(n_str)}")


open = []

initial = list(GG.nodes)[0]
n = initial


neighbors = list(undirected.neighbors(initial))
if len(neighbors) == 2:
    n1 = neighbors[0]
    n2 = neighbors[1]

    # add (n, n1) and (prime(n1), prime(n)) to the graph
    add_oriented_edge(n, n1)
    # add (n2, n) and (prime(n), prime(n2)) to the graph
    add_oriented_edge(n2, n)
elif len(neighbors) == 3:
    close_pair = set(closest_neighbors(G_proj, n))
    for nn in close_pair:
        # add (n, nn) and (prime(nn), prime(n)) to the graph
        # closest_neighbors() can hand back float-typed ids - normalize to int
        # so the GGG node id matches the plain str(n) ids used everywhere else
        add_oriented_edge(n, int(nn))
    for nn in neighbors:
        if nn in close_pair:
            continue
        # add (nn, n) and (prime(n), prime(nn)) to the graph
        add_oriented_edge(nn, n)
else:
    for nn in neighbors:
        # add (n, nn) and (prime(nn), prime(n)) to the graph
        add_oriented_edge(n, nn)


open = neighbors
closed = {initial}

print(initial)

while len(open) > 0:
    
    n = open[0]
    open = open[1:]
    if n in closed:
        continue

    neighbors = list(undirected.neighbors(n))
    closed.add(n)

    if len(neighbors) == 1:
        print(f"{n} c1")
        # open only contains reachable nodes, so nothing left to do.
        continue

    elif len(neighbors) == 2:
        print(f"{n} c2")
        # open only contains reachable nodes, so one existing and other new

        # assert exactly one neighbor in closed, find which one
        existing = [x for x in neighbors if x in closed]
        new = [x for x in neighbors if x not in closed]
        if len(existing) == 2:
            nn1 = neighbors[0]
            nn2 = neighbors[1]

            # assert at least one candidate is already in GGG, take one of those
            nnn1_candidates = [str(x) for x in find_other_neighbor(undirected, nn1, n)]
            ggg_undirected = GGG.to_undirected()
            nnn1_in_ggg = [x for x in nnn1_candidates if ggg_undirected.has_edge(x, str(nn1))]

            if not nnn1_in_ggg:
                print(f"postpone {n}")
                open.append(n)
                closed.remove(n)
                continue
            
            # assert nnn1_in_ggg, f"none of {nnn1_candidates} is in GGG"
            nnn1 = nnn1_in_ggg[0]

            nnn2_candidates = [str(x) for x in find_other_neighbor(undirected, nn2, n)]
            nnn2_in_ggg = [x for x in nnn2_candidates if ggg_undirected.has_edge(x, str(nn2))]
            if not nnn2_in_ggg:
                print(f"postpone {n}")
                open.append(n)
                closed.remove(n)
                continue
            assert nnn2_in_ggg, f"none of {nnn2_candidates} is in GGG"
            nnn2 = nnn2_in_ggg[0]

            v1 = find_me(nnn1,str(nn1))
            v2 = find_me(str(nn2),nnn2)

            add_oriented_edge(v1, str(n))
            add_oriented_edge(str(n), v2)

            add_oriented_edge(prime(v2), prime(str(n)))
            add_oriented_edge(prime(str(n)), prime(v1))
            
            continue
        elif len(existing) == 0:
            raise        
        
        assert len(existing) == 1, f"expected exactly one closed neighbor of {n}, got {existing}"
        assert len(new) == 1, f"expected exactly one new neighbor of {n}, got {new}"
        existing_neighbor = existing[0]
        new_neighbor = new[0]

        en = str(existing_neighbor)
        n_str = str(n)
        me = find_me(en, n_str)

        add_oriented_edge(me, str(new_neighbor))
        add_oriented_edge(prime(str(new_neighbor)), prime(me))

        open.append(new_neighbor)

    elif len(neighbors) == 3:
        print(f"{n} c3")

        # assert one neighbor existing two new, find out which ones
        existing = [x for x in neighbors if x in closed]
        new = [x for x in neighbors if x not in closed]
        if len(existing) == 2:
            print(f"{n} c3")
            close_pair = set(closest_neighbors(G_proj, n))
            existing_close = [x for x in existing if x in close_pair]
            new_neighbor = new[0]
            n_str = str(n)
            if len(existing_close) == 2:
                # case 1: 2 existing are both close neighbors
                me = find_me(str(existing[0]), n_str)
                assert me == find_me(str(existing[1]), n_str), f"close-pair legs of switch {n} resolved to different identities"
            elif len(existing_close) == 1:
                # case 2: 1 existing is close neighbor, the other not close
                existing_point = [x for x in existing if x not in close_pair][0]
                me = find_me(str(existing_point), n_str)
            else:
                raise ValueError(f"unexpected close_pair/existing combination at switch {n}: existing={existing}, close_pair={close_pair}")

            # add_oriented_edge(me, str(new_neighbor))
            # add_oriented_edge(prime(str(new_neighbor)), prime(me))
            # open.append(new_neighbor)
            continue
        assert len(existing) == 1, f"expected exactly one closed neighbor of {n}, got {existing}"
        assert len(new) == 2, f"expected exactly two new neighbors of {n}, got {new}"
        existing_neighbor = existing[0]
        new_neighbor_1, new_neighbor_2 = new

        en = str(existing_neighbor)
        n_str = str(n)
        me = find_me(en, n_str)

        for new_neighbor in new:
            add_oriented_edge(me, str(new_neighbor))
            add_oriented_edge(prime(str(new_neighbor)), prime(me))
            open.append(new_neighbor)
    else:
        raise

In [ ]:
# verify implementation 1 (BFS with inline degree-based case analysis)
verify_edge_duality(GGG)


In [ ]:
# verify implementation 1 (BFS with inline degree-based case analysis)
verify_ggg_paths(GGG)


In [ ]:
# # go through all nodes. cluster all neighbors into whether they are plus or minus
# # - if neighbors == 2, assign one undirected neighbor to plus and the other to minus
# # - if neighbors == 3, use p as "spitz" (point end) coming from n to the close
# #   neighbors n1,n2 (see closest_neighbors()); i.e. assign n1,n2 to plus and p to minus
# # all keys and set values are kept as str so they line up with GGG's (str / "_prime")
# # node ids
# plus = {}
# minus = {}



# for n in GG.nodes:
#     neighbors = list(undirected.neighbors(n))
#     if len(neighbors) == 2:
#         plus[str(n)] = {str(neighbors[0])}
#         minus[str(n)] = {str(neighbors[1])}
#     elif len(neighbors) == 3:
#         close_pair = set(closest_neighbors(G_proj, n))
#         plus[str(n)] = {str(nb) for nb in neighbors if nb in close_pair}
#         minus[str(n)] = {str(nb) for nb in neighbors if nb not in close_pair}
#     else:
#         # boundary node (degree <= 1): nothing to split
#         plus[str(n)] = {str(nb) for nb in neighbors}
#         minus[str(n)] = set()

#     # the "_prime" twin is the reverse-direction copy of n, so its plus/minus sides
#     # are swapped relative to n's - this is what makes
#     # "n2 in plus[n1] <=> prime(n2) in minus[prime(n1)]" hold (see cell below)
#     plus[prime(str(n))] = {prime(x) for x in minus[str(n)]}
#     minus[prime(str(n))] = {prime(x) for x in plus[str(n)]}

#     print(n, "plus:", plus[str(n)], "minus:", minus[str(n)])
#     print(prime(str(n)), "plus:", plus[prime(str(n))], "minus:", minus[prime(str(n))])

In [ ]:
# GGG = nx.DiGraph()
# GGG.graph["crs"] = GG.graph.get("crs")


# undirected = GG.to_undirected()


# for beam in boundary_paths:
#     print(f"beam {beam}")
#     previous = None
#     for n1, n2 in zip(beam, beam[1:]):
#         n1_prime = f"{n1}_prime"
#         n2_prime = f"{n2}_prime"
        
#         if previous is None:
#             if str(n2) in plus[str(n1)] or prime(str(n2)) in minus[prime(str(n1))]:
#                 previous = str(n1)
#             elif str(n2) in minus[str(n1)]:
#                 previous = n1_prime
#             else:
#                 raise
#         else:
#             print(previous,n1,n2,)
            
#             # `previous` encodes the physical node we came from (plus: plain id,
#             # minus: its "_prime" twin) - recover that physical id to look up its
#             # GG node/edge attributes
#             prev_id = int(previous[:-len("_prime")]) if previous.endswith("_prime") else int(previous)
#             edge_data = GG.get_edge_data(prev_id, n1) or GG.get_edge_data(n1, prev_id) or {}

#             if str(n1) in plus[previous]:
#                 print(f" add {previous, str(n1)}")
                
#                 # add edges (previous, n1) and (n1_prime, prime(previous)) to the graph
#                 GGG.add_node(previous, **GG.nodes[prev_id])
#                 GGG.add_node(str(n1), **GG.nodes[n1])
#                 assert previous != str(n1), f"self-loop edge {previous} -> {str(n1)}"
#                 GGG.add_edge(previous, str(n1), **edge_data)

#                 GGG.add_node(n1_prime, **GG.nodes[n1])
#                 GGG.add_node(prime(previous), **GG.nodes[prev_id])
#                 assert n1_prime != prime(previous), f"self-loop edge {n1_prime} -> {prime(previous)}"
#                 GGG.add_edge(n1_prime, prime(previous), **edge_data)

#                 previous = str(n1)
#             elif prime(str(n1)) in plus[previous]:
#                 print(f" add {previous, n1_prime}")
                
#                 # add edges (previous, n1_prime) and (n1, prime(previous)) to the graph
#                 GGG.add_node(previous, **GG.nodes[prev_id])
#                 GGG.add_node(n1_prime, **GG.nodes[n1])
#                 print(n1,n2,previous)
#                 assert previous != n1_prime, f"self-loop edge {previous} -> {n1_prime}"
#                 GGG.add_edge(previous, n1_prime, **edge_data)

#                 GGG.add_node(str(n1), **GG.nodes[n1])
#                 GGG.add_node(prime(previous), **GG.nodes[prev_id])
#                 assert str(n1) != prime(previous), f"self-loop edge {str(n1)} -> {prime(previous)}"
#                 GGG.add_edge(str(n1), prime(previous), **edge_data)

#                 previous = n1_prime
#             else:
                
#                 raise    

In [ ]:
list(GGG.successors("2042222934"))

In [ ]:
list(GGG.predecessors("2042222928"))

In [ ]:
list(GGG.successors("2042222934"))

## General, order-independent construction of GGG


In [ ]:
# General, order-independent construction of GGG from GG's undirected version:
# orient every physical edge exactly once such that, at every switch, the
# close, near-collinear neighbor pair (its "legs" - see closest_neighbors())
# always ends up on the SAME side (both in, or both out) while the third
# neighbor (the "point") always ends up on the OPPOSITE side - that is the
# only constraint. Non-switch nodes simply alternate in/out between their two
# neighbors. Which side (in vs out) a switch/node actually settles on is only
# decided once one of its edges is already oriented (never fixed upfront) -
# this is what correctly handles a switch directly adjacent to another switch
# (e.g. a crossover), where each switch's own close pair may independently
# want to treat the shared edge as a leg: whichever side gets oriented first
# determines the other via this same propagation, instead of the two
# independent per-switch choices conflicting.
#
# Because a switch's two legs always share one role and its point always
# takes the other, there is never an edge from one leg directly to the other
# through the switch - the only way through is point <-> leg (and, via the
# "_prime" mirror, leg_prime <-> point_prime) - so no path can ever go over
# the small angle in the switch (n1 -> s -> n2 for close neighbors n1, n2).
# This also guarantees every boundary_paths path in GG has a corresponding
# directed path in GGG, since boundary_paths already excludes leg-to-leg
# steps using the same close_neighbor_pairs.

GGG = nx.DiGraph()
GGG.graph["crs"] = GG.graph.get("crs")

edge_orientation = {}  # frozenset({a, b}) -> (src, dst)

def orient(src, dst):
    key = frozenset((src, dst))
    if key in edge_orientation:
        assert edge_orientation[key] == (src, dst), (
            f"conflicting orientation for edge {src}-{dst}: "
            f"already {edge_orientation[key]}, now requested {(src, dst)}"
        )
        return False
    edge_orientation[key] = (src, dst)
    return True


def propagate(queue):
    """Whenever one of a node's incident edges is already oriented, derive
    the rest: a plain node alternates in/out between its two edges; a switch
    keeps its two close-pair legs on the same side and its point on the
    opposite side."""
    while queue:
        n = queue.popleft()
        neighbors = list(undirected.neighbors(n))
        known = {
            nb: edge_orientation[frozenset((n, nb))]
            for nb in neighbors
            if frozenset((n, nb)) in edge_orientation
        }
        if not known or len(known) == len(neighbors):
            continue  # nothing to derive yet, or already fully resolved

        if len(neighbors) == 2:
            kn, kd = next(iter(known.items()))
            un = next(nb for nb in neighbors if nb not in known)
            into_n = kd == (kn, n)
            if orient(*((un, n) if not into_n else (n, un))):
                queue.append(un)

        elif len(neighbors) == 3:
            close_pair = close_neighbor_pairs[n]
            role = {nb: ("leg" if nb in close_pair else "point") for nb in neighbors}

            leg_into_n = next(
                (d == (nb, n)) if role[nb] == "leg" else not (d == (nb, n))
                for nb, d in known.items()
            )
            for nb in neighbors:
                want_into_n = leg_into_n if role[nb] == "leg" else not leg_into_n
                if orient(*((nb, n) if want_into_n else (n, nb))):
                    queue.append(nb)

        else:
            raise ValueError(f"unexpected degree {len(neighbors)} at node {n}")


# pick any still-unoriented edge, orient it arbitrarily, and flood-fill the
# rest of its connected component via propagate() - repeat until every edge
# of GG has been oriented
for a, b in undirected.edges:
    if frozenset((a, b)) not in edge_orientation:
        orient(a, b)
        propagate(deque([a, b]))

assert len(edge_orientation) == undirected.number_of_edges(), "not all edges of GG were oriented"

for src, dst in edge_orientation.values():
    add_oriented_edge(src, dst)


In [ ]:
# verify implementation 2 (general, order-independent edge-orientation construction)
verify_edge_duality(GGG)


In [ ]:
# verify implementation 2 (general, order-independent edge-orientation construction)
verify_ggg_paths(GGG)


## 3rd implementation: single-pass BFS, close-pair check folded into both branches


In [ ]:
# 3rd implementation: single-pass BFS (open/closed), generalizing the degree-2
# and degree-3 handling into one rule: whichever already-wired neighbor of n
# is found first fixes n's established identity ("me"); for switches, that
# neighbor's leg/point group then also fixes the required role (into/out of
# me) for every other neighbor - keeping the two legs on the same side and
# the point on the opposite side. This close-pair check is folded into BOTH
# the "neighbor already wired elsewhere" (merge, postponed if not yet
# resolvable) and "neighbor still fresh" cases, so a switch's rule is honored
# no matter which of its neighbors gets discovered first - including a
# switch directly adjacent to another switch (e.g. a crossover).
#
# Caveat: unlike the propagate()-based 2nd implementation, this doesn't
# globally verify consistency as it goes, so a topologically-impossible
# junction (e.g. the 3-switch triangular loop discussed above) won't raise -
# it will just get resolved one particular way depending on BFS order, and
# the no-leg-to-leg guarantee may not hold for that specific junction while
# still holding everywhere else.

GGG = nx.DiGraph()
GGG.graph["crs"] = GG.graph.get("crs")
undirected = GG.to_undirected()


def group_role(n, nb):
    """'leg' or 'point' for neighbor `nb` of switch `n`; None if n isn't a switch."""
    if n not in close_neighbor_pairs:
        return None
    return "leg" if nb in close_neighbor_pairs[n] else "point"


def edge_into(target, source_candidate):
    """True if source_candidate (or its prime) already has an edge into
    `target` in GGG; False if `target` already has an edge out to it."""
    sc = str(source_candidate)
    if (sc, target) in GGG.edges or (prime(sc), target) in GGG.edges:
        return True
    elif (target, sc) in GGG.edges or (target, prime(sc)) in GGG.edges:
        return False
    else:
        raise ValueError(f"no edge found between {target} and {sc}/{prime(sc)}")


def desired_into_me(n, me, known_neighbor, target):
    """Whether `target` should feed INTO `me` (True) or receive FROM `me`
    (False), given `known_neighbor` is n's already-established neighbor."""
    known_into_me = edge_into(me, known_neighbor)
    if n not in close_neighbor_pairs:
        return not known_into_me  # plain node: the other neighbor takes the opposite role
    same_group = group_role(n, known_neighbor) == group_role(n, target)
    return known_into_me if same_group else not known_into_me


open = deque()

initial = list(GG.nodes)[0]
neighbors = list(undirected.neighbors(initial))
if len(neighbors) == 2:
    add_oriented_edge(initial, neighbors[0])
    add_oriented_edge(neighbors[1], initial)
elif len(neighbors) == 3:
    close_pair = close_neighbor_pairs[initial]
    point = next(x for x in neighbors if x not in close_pair)
    add_oriented_edge(point, initial)
    for leg in close_pair:
        # closest_neighbors() can hand back float-typed ids - normalize to int
        add_oriented_edge(initial, int(leg))
else:
    raise ValueError(f"unexpected degree {len(neighbors)} at initial node {initial}")

open.extend(neighbors)
closed = {initial}

while open:
    n = open.popleft()
    if n in closed:
        continue

    neighbors = list(undirected.neighbors(n))
    ggg_undirected = GGG.to_undirected()
    existing = [nb for nb in neighbors
                if ggg_undirected.has_edge(str(n), str(nb)) or ggg_undirected.has_edge(prime(str(n)), str(nb))]
    assert existing, f"node {n} visited with no existing GGG connection"

    known_neighbor = existing[0]
    me = find_me(str(known_neighbor), str(n))
    for other in existing[1:]:
        assert find_me(str(other), str(n)) == me, (
            f"node {n}'s existing neighbors resolve to different identities"
        )

    closed.add(n)
    postponed = False
    for nn in neighbors:
        if nn in existing:
            continue
        want_into_me = desired_into_me(n, me, known_neighbor, nn)

        if str(nn) in GGG.nodes:
            # nn already wired elsewhere - find which of its identities is
            # already resolved, via whichever of its other neighbors it's
            # already connected to
            nn_candidates = [str(x) for x in find_other_neighbor(undirected, nn, n)]
            ggg_undirected = GGG.to_undirected()
            nn_in_ggg = [x for x in nn_candidates if ggg_undirected.has_edge(x, str(nn))]
            if not nn_in_ggg:
                print(f"postpone {n}")
                open.append(n)
                closed.remove(n)
                postponed = True
                break
            v = find_me(nn_in_ggg[0], str(nn))
            if want_into_me:
                add_oriented_edge(v, me)
            else:
                add_oriented_edge(me, v)
        else:
            if want_into_me:
                add_oriented_edge(str(nn), me)
            else:
                add_oriented_edge(me, str(nn))
            open.append(nn)

    if not postponed:
        print(n)


In [ ]:
# verify implementation 3 (single-pass BFS, close-pair folded into both branches)
verify_edge_duality(GGG)


In [ ]:
# verify implementation 3 (single-pass BFS, close-pair folded into both branches)
verify_ggg_paths(GGG)


## Display GGG as interactive map

In [ ]:
# same categories as above (see "Visual inspection" / interactive map section)
explore_categorized(GGG)

In [ ]:
len(GGG.edges())

In [ ]:
for e in GGG.edges():
    print(e)

In [ ]:
import networkx as nx
nx.draw(GGG, labels={n[0]: str(n[0]) for n in list(GGG.nodes(data=True))})

## Results: which GGG-construction approach passes which verification

| Implementation | Graph | Completes? | `verify_edge_duality` | `verify_ggg_paths` |
|---|---|---|---|---|
| **1** — BFS, inline degree-based case analysis | original (`G`) | crashes (`ValueError`, no edge found) | — | — |
| **1** | `G_fixed` | yes | pass | **fail** |
| **2** — global `orient`/`propagate` flood-fill | original (`G`) | crashes (`AssertionError` at the [triangular junction](https://www.openstreetmap.org/node/2042222928)) | — | — |
| **2** | `G_fixed` | yes | pass | **pass** |
| **3** — single-pass BFS, close-pair check folded into both branches | original (`G`) | yes | pass | fail (only at the [triangular junction](https://www.openstreetmap.org/node/2042222928)) |
| **3** | `G_fixed` | yes | pass | **pass** |

### Why "locality" is not enough

Implementations 1-3 all decide a node's two `GGG` identities (plain / `_prime`) using only *locally* available
information: the node's immediate neighbors, and - for switches - which neighbor pair is the close (leg) pair.
This is enough almost everywhere, because the doubling scheme only has one degree of freedom per plain/undirected
edge (which of its two directed orientations is "into" vs "out of" a given node), and propagating that choice
along a path of degree-2 nodes and ordinary switches never conflicts with itself.

It breaks down on an **odd cycle of switches**: three switches connected pairwise (e.g. the triangular/wye
junction at [2042222928](https://www.openstreetmap.org/node/2042222928), switches
[2042222934](https://www.openstreetmap.org/node/2042222934),
[2042222937](https://www.openstreetmap.org/node/2042222937),
[2042222947](https://www.openstreetmap.org/node/2042222947)). Walking around the loop, each switch "flips" which
of its two neighbors is the leg-into vs. leg-out-of side relative to the point. An odd number of flips around a
closed loop is self-contradictory (same parity problem as 2-coloring an odd cycle) - **no choice of local
propagation order, and no amount of backtracking, can satisfy all three switches' leg/point constraints
simultaneously** if all three are mutually adjacent with no node in between. This is a structural property of the
graph topology, not a bug in any one implementation - it's why implementation 2 crashes loudly (it checks global
consistency and asserts), why implementation 3 silently produces an inconsistent path through that junction (it
never cross-checks consistency globally, only locally during its single BFS pass), and why implementation 1 never
even gets that far (it crashes earlier on an unrelated, still-unfixed gap in its own degree-3 case analysis that
ignores close-pairs for the "1 existing neighbor" branch).

### Why we need to fix the switch-switch edges

The odd cycle only exists because OSMnx's default `simplify_graph` collapses every interstitial (non-junction)
node on a way, including between two adjacent switches - leaving the three switches of the triangular junction
directly pairwise adjacent, with no "neutral" node between any two of them to absorb the parity conflict. Once a
single intermediate node is restored on just one of the triangle's three switch-switch edges (see
`simplify_graph_keep_switch_intermediate` above, applied to the non-simplified graph `G_non_simple`), that edge is
no longer a direct switch-to-switch edge - the loop is broken, the parity constraint disappears, and both
implementation 2 and implementation 3 pass both verifications on `G_fixed`. This is therefore a **data-level**
fix (recovering OSM information that `simplify_graph` is discarding), not an algorithmic one - no purely
algorithmic fix to the construction logic can resolve a genuine odd cycle of mutually-adjacent switches.

Implementation 1's remaining failure on `G_fixed` is unrelated to the triangular junction; it is the pre-existing
gap in its degree-3/"1 existing neighbor" branch, which never checks close-neighbor pairs (unlike implementations
2 and 3) and has not been fixed.
